# <u>Step 7 - Lesson 3</u> :- **Deep Q-Networks (DQN) from Scratch**

In Lessons 1 & 2, we learned PyTorch fundamentals (layers, forward passes, MSELoss, gradient descent).
Now we take the **2D GridWorld Q-Learning** from `proj.py` and replace the dictionary `Q = {(r, c): {...}}` with a **Deep Neural Network**!

### Why Deep Q-Learning?
A Q-table needs an entry for every state. In complex games, continuous robotics, or rocket landing, the number of states is infinite. A Neural Network acts as a **function approximator** predicting Q-values for any state `(r, c)`.

### The 2 Secret Weapons of DQN:
1. **Experience Replay Buffer (`deque`)**: Breaks correlation between consecutive steps by randomly sampling past `(state, action, reward, next_state, done)` transitions.
2. **Target Network (`target_net`)**: Freezes the target calculation weights to prevent the "moving target problem" and stabilizes training.

## Step 1 :- Import Libraries

In [1]:
import random
from collections import deque
import torch
import torch.nn as nn
import torch.optim as optim

## Step 2 :- Neural Network Architecture (`DQN`)

- **Input**: State coordinates `(row, col)` (dimension = 2)
- **Hidden Layers**: Two fully connected linear layers of 64 neurons with `ReLU` activations
- **Output**: 4 Q-values `[Q(s, left), Q(s, right), Q(s, up), Q(s, down)]`

In [2]:
class DQN(nn.Module):
    def __init__(self, state_dim=2, action_dim=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim)
        )

    def forward(self, x):
        return self.net(x)

action_names = ['left', 'right', 'up', 'down']

## Step 3 :- 2D GridWorld Environment

- 3x3 Grid from `(0, 0)` to `(2, 2)`
- **Goal**: `(2, 2)` with Reward `+100.0` and `done = True`
- **Step penalty**: `-1.0`
- **Boundary Violation**: Resets back to `(0, 0)`

In [3]:
def step(state, action_idx):
    action = action_names[action_idx]
    
    if action == 'left':
        next_state = (state[0], state[1] - 1)
    elif action == 'right':
        next_state = (state[0], state[1] + 1)
    elif action == 'up':
        next_state = (state[0] + 1, state[1])
    else:  # 'down'
        next_state = (state[0] - 1, state[1])
        
    # Out of bounds reset (portal mechanic)
    if next_state[0] < 0 or next_state[1] < 0 or next_state[0] > 2 or next_state[1] > 2:
        next_state = (0, 0)
        
    # Goal check (2, 2)
    if next_state[0] == 2 and next_state[1] == 2:
        reward = 100.0
        done = True
    else:
        reward = -1.0
        done = False
        
    return next_state, reward, done

## Step 4 :- Hyperparameters, Target Network & Replay Buffer

- `q_net`: The active network being trained.
- `target_net`: Copied weights from `q_net`, updated periodically (every 10 episodes) to stabilize TD targets.
- `replay_buffer`: Holds up to 5,000 transitions; batches of 32 are sampled randomly.

In [4]:
q_net = DQN()
target_net = DQN()
target_net.load_state_dict(q_net.state_dict())

optimizer = optim.Adam(q_net.parameters(), lr=0.001)
criterion = nn.MSELoss()

replay_buffer = deque(maxlen=5000)
batch_size = 32
gamma = 0.9
epsilon = 0.2

## Step 5 :- Action Selection ($\epsilon$-Greedy with PyTorch)

With probability $\epsilon$, pick a random action index `[0..3]`. Otherwise, pass the state through `q_net` and choose $\arg\max Q(s, a)$.

In [5]:
def choose_action(state, epsilon):
    if random.random() < epsilon:
        return random.randint(0, 3)

    with torch.no_grad():
        state_t = torch.tensor(state, dtype=torch.float32)
        q_values = q_net(state_t)
        return torch.argmax(q_values).item()

## Step 6 :- Training Loop with Experience Replay & Bellman Loss

For each transition:
1. Store `(state, action, reward, next_state, done)` into `replay_buffer`.
2. Sample a random mini-batch of 32 transitions.
3. Compute Bellman Target using `target_net`:
   $$\text{Target} = R + \gamma \max_{a'} Q_{\text{target}}(s', a') \times (1 - \text{done})$$
4. Compute loss between predicted $Q(s, a)$ from `q_net` and target, then backpropagate.
5. Every 10 episodes, sync `target_net` weights with `q_net`.

In [6]:
print("Training the Network :-")

for i in range(500):
    state = (0, 0)
    done = False

    while not done:
        action = choose_action(state, epsilon)
        next_state, reward, done = step(state, action)

        replay_buffer.append((state, action, reward, next_state, done))
        state = next_state

        if len(replay_buffer) >= batch_size:
            batch = random.sample(replay_buffer, batch_size)

            states      = torch.tensor([b[0] for b in batch], dtype=torch.float32)
            actions     = torch.tensor([b[1] for b in batch], dtype=torch.long)
            rewards     = torch.tensor([b[2] for b in batch], dtype=torch.float32)
            next_states = torch.tensor([b[3] for b in batch], dtype=torch.float32)
            dones       = torch.tensor([b[4] for b in batch], dtype=torch.float32)

            q_values = q_net(states)
            predictions = q_values.gather(1, actions.unsqueeze(1)).squeeze(1)

            with torch.no_grad():
                next_q = target_net(next_states).max(dim=1)[0]
                targets = rewards + gamma * next_q * (1.0 - dones)

            loss = criterion(predictions, targets)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    if i % 10 == 0:
        target_net.load_state_dict(q_net.state_dict())

print("Training finished!\n")

Training the Network :-
Training finished!


## Step 7 :- Test the Trained DQN Policy Across the Grid

We query `q_net` for every tile `(row, col)` in the 3x3 grid to inspect the learned policy map.

In [7]:
print("--- Trained Policy (Best Action from each Tile) ---")
grid_policy = []
for r in range(3):
    row_actions = []
    for c in range(3):
        if (r, c) == (2, 2):
            row_actions.append("GOAL")
            continue
        with torch.no_grad():
            st = torch.tensor([r, c], dtype=torch.float32)
            best_a = torch.argmax(q_net(st)).item()
            row_actions.append(action_names[best_a])
    grid_policy.append(row_actions)

for row in grid_policy:
    print(row)

--- Trained Policy (Best Action from each Tile) ---
['up', 'right', 'up']
['right', 'right', 'up']
['right', 'right', 'GOAL']


## Step 8 :- Walk the Learned Path (Simulation)

Starting at `(0, 0)`, let the trained DQN steer directly to the goal with greedy decisions ($\epsilon = 0$):

In [8]:
print("Starting Path Simulation from (0, 0) :-")
curr_s = (0, 0)
path_done = False
steps_count = 0
total_sim_reward = 0

while not path_done and steps_count < 10:
    with torch.no_grad():
        st_tensor = torch.tensor(curr_s, dtype=torch.float32)
        best_act_idx = torch.argmax(q_net(st_tensor)).item()
    nxt_s, rew, path_done = step(curr_s, best_act_idx)
    print(f"At {curr_s} -> Action: {action_names[best_act_idx]:<5} -> Next State: {nxt_s} | Reward: {rew}")
    total_sim_reward += rew
    curr_s = nxt_s
    steps_count += 1

print(f"\nFinal State Reached: {curr_s}")
print(f"Total Steps: {steps_count} | Total Reward: {total_sim_reward}")

Starting Path Simulation from (0, 0) :-
At (0, 0) -> Action: up    -> Next State: (1, 0) | Reward: -1.0
At (1, 0) -> Action: right -> Next State: (1, 1) | Reward: -1.0
At (1, 1) -> Action: right -> Next State: (1, 2) | Reward: -1.0
At (1, 2) -> Action: up    -> Next State: (2, 2) | Reward: 100.0

Final State Reached: (2, 2)
Total Steps: 4 | Total Reward: 97.0
